# 모델 성능을 높이는 프롬프트 기법 
- 모델에게 정확한 지시를 제공하고, 원하는 출력을 얻기 위해 입력을 최적화하는 기술
- [프롬프트 엔지니어닝 가이드](https://www.promptingguide.ai/kr)

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI 
from dotenv import load_dotenv
import os

In [6]:
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv(dotenv_path='../.env')

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print(OPENAI_API_KEY[:2])


gs


In [7]:
# # Groq API를 사용하는 ChatOpenAI 인스턴스 생성
llm = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    base_url="https://api.groq.com/openai/v1",  # Groq API 엔드포인트
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    temperature=0.7
)

# llm = ChatOpenAI(api_key=OPENAI_API_KEY, model_name="gpt-4o")
print(llm)

client=<openai.resources.chat.completions.completions.Completions object at 0x00000182919FC1A0> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000182919FE4B0> root_client=<openai.OpenAI object at 0x00000182919D02C0> root_async_client=<openai.AsyncOpenAI object at 0x000001829140A000> model_name='meta-llama/llama-4-scout-17b-16e-instruct' temperature=0.7 model_kwargs={} openai_api_key=SecretStr('**********') openai_api_base='https://api.groq.com/openai/v1'


### 1) 구체적이고 명확하게 사용자의 지시를 제공합니다.
- 프롬프트는 모델이 이해하기 쉽게 명확하고 간결해야 합니다.  
- 불필요한 정보를 줄이고, 핵심 요구 사항에 집중해야 합니다.
- 원하는 출력이 무엇인지 모델에게 정확하게 알려주어야 합니다. 

In [13]:
# 질문 (Question)
template_text = "How many parameters does the {model_name} model have? Please display the number in the format 1,000,000)"
#"{model_name} 모델의 파라미터 개수는 몇개인가요? 답변은 아라비아 숫자만을 사용해서 답변해주세요. "

prompt_template = PromptTemplate.from_template(template_text)

models = [
    {"model_name": "gpt-3.5-turbo"},
    {"model_name": "gpt-4o"},
    {"model_name": "mistral-saba-24b"},
    {"model_name": "llama-4-scout-17b-16e-instruct"},
]  

# 여러 개의 프롬프트를 미리 생성
formatted_prompts = [prompt_template.format(**q) for q in models]
print(formatted_prompts)  # 미리 생성된 질문 목록 확인

for idx, prompt in enumerate(formatted_prompts,1):
    response = llm.invoke(prompt)
    print(f'==> {idx} <==')
    print(response.content)

['How many parameters does the gpt-3.5-turbo model have? Please display the number in the format 1,000,000)', 'How many parameters does the gpt-4o model have? Please display the number in the format 1,000,000)', 'How many parameters does the mistral-saba-24b model have? Please display the number in the format 1,000,000)', 'How many parameters does the llama-4-scout-17b-16e-instruct model have? Please display the number in the format 1,000,000)']
==> 1 <==
The gpt-3.5-turbo model, also known as the ChatGPT model, has approximately 175 billion parameters.

Here is the number in the requested format: 175,000,000,000 

or 

 175,000,000,000) 
 I assume you meant to type ( so here is 

175,000,000,000)
==> 2 <==
The GPT-4o model, like other GPT models, is a large language model with a significant number of parameters. According to OpenAI, GPT-4o has **1,000,000,000** parameters, which can also be expressed as **1 billion** parameters.

So, here is the number in the requested format: **1,000

### 2) 참고할 수 있는 예시를 제공합니다.
- 원하는 출력 형식이나 스타일을 모델에게 보여주기 위해 예시를 사용할 수 있습니다. 
- 이는 모델이 출력의 방향을 잡는데 도움이 됩니다.

In [14]:
# 컨텍스트 제공 (Context)

response = llm.invoke(
    """다음 제시된 뉴스를 기반으로 질문에 답변하세요:
    뉴스: 삼성전자가 내년 초에 자체적으로 개발한 인공지능(AI) 가속기를 처음으로 출시할 예정이다. 
          이는 AI 반도체 시장에서 지배적인 위치를 차지하고 있는 엔비디아의 독점에 도전하고, 
          세계 최고의 반도체 제조업체로서의 지위를 다시 확립하려는 삼성전자의 노력으로 해석된다.
    
    질문: AI 반도체 시장에서 지배적인 위치를 차지하고 있는 회사는 어디인가요? 회사이름만 출력해주세요.
    답변:
    """)
print(response.content)

엔비디아


In [15]:
# 예시 없음 (zero-shot)

response = llm.invoke(
    """다음 제시된 뉴스에서 3개의 키워드를 추출하세요:
    뉴스: 삼성전자가 내년 초에 자체적으로 개발한 인공지능(AI) 가속기를 처음으로 출시할 예정이다. 
          이는 AI 반도체 시장에서 지배적인 위치를 차지하고 있는 엔비디아의 독점에에 도전하고, 
          세계 최고의 반도체 제조업체로서의 지위를 다시 확립하려는 삼성전자의 노력으로 해석된다.
    
    키워드: 
    """)
print(response.content)

삼성전자, 인공지능, 엔비디아


In [16]:
# 1개의 예시를 제공 (one-shot)

response = llm.invoke(
    """다음 예시와 같이 제시된 뉴스에서 3개의 키워드를 추출하세요:
    <예시>
    뉴스: 삼성전자가 내년 초에 자체적으로 개발한 인공지능(AI) 가속기를 처음으로 출시할 예정이다. 
          이는 AI 반도체 시장에서 지배적인 위치를 차지하고 있는 엔비디아의 독점을 도전하고, 
          세계 최고의 반도체 제조업체로서의 지위를 다시 확립하려는 삼성전자의 노력으로 해석된다.
    
    키워드: 삼성전자, 인공지능, 엔비디아
    </예시>

    AI의 영향을 가장 크게 받은 구글 제품은 바로 구글 검색입니다. 
    현재 10억 명의 이용자가 구글의 AI 개요(AI Overviews) 기능을 통해 완전히 새로운 유형의 질문을 할 수 있게 됐으며, 이는 가장 인기 있는 검색 기능 중 하나가 됐습니다. 
    구글은 다음 단계로, 제미나이 2.0의 고급 추론 기능을 AI 개요에 적용해 고급 수학 방정식, 멀티모달 쿼리 및 코딩 등 더 복잡한 질문을 처리할 수 있도록 개선할 예정입니다. 
    구글은 이번 주에 제한된 범위의 테스트를 시작했으며 내년 초에 더 광범위하게 출시할 예정입니다. 
    또한 내년에는 AI 개요 기능을 더 많은 국가와 언어로 확대해 선보일 계획입니다.
    
    키워드:
    """)
print(response.content)

구글, 인공지능, 제미나이


In [17]:
# 여러 개의 예시를 제공 (few-shot)
response = llm.invoke(
    """다음 예시들과 같이 제시된 뉴스에서 각각 3개의 키워드를 추출하세요:
    <예시1>
    뉴스: 삼성전자가 내년 초에 자체적으로 개발한 인공지능(AI) 가속기를 처음으로 출시할 예정이다. 
          이는 AI 반도체 시장에서 지배적인 위치를 차지하고 있는 엔비디아의 독점을 도전하고, 
          세계 최고의 반도체 제조업체로서의 지위를 다시 확립하려는 삼성전자의 노력으로 해석된다.
    키워드: 삼성전자, 인공지능, 엔비디아
    </예시1>

    <예시2>
    뉴스: 세계보건기구(WHO)는 최근 새로운 건강 위기에 대응하기 위해 국제 협력의 중요성을 강조했다. 
          전염병 대응 역량의 강화와 글로벌 보건 시스템의 개선이 필요하다고 발표했다.
    키워드: 세계보건기구 | 건강위기 | 국제 
    </예시2>

    뉴스: 제미나이 2.0 플래시는 현재 구글 AI 스튜디오(Google AI Studio) 및 버텍스 AI(Vertex AI) 에서 제미나이 API를 통해 개발자에게 실험 모델로 제공됩니다. 
         모든 개발자는 멀티모달 입력 및 텍스트 출력을 사용할 수 있으며, 텍스트 음성 변환(text-to-speech) 및 네이티브 이미지 생성은 일부 파트너들을 대상으로 제공됩니다. 
         내년 1월에는 더 많은 모델 사이즈와 함께 일반에 공개될 예정입니다.
      
     키워드:
    """
)
# gpt-4o
# 키워드: 제미나이 2.0, 구글 AI 스튜디오, 버텍스 AI 
print(response.content)

뉴스: 제미나이2.0 플래시는 현재 구글 AI 스튜디오(Google AI Studio) 및 버텍스 AI(Vertex AI) 에서 제미나이 API를 통해 개발자에게 실험 모델로 제공됩니다. 
모든 개발자는 멀티모달 입력 및 텍스트 출력을 사용할 수 있으며, 텍스트 음성 변환(text-to-speech) 및 네이티브 이미지 생성은 일부 파트너들을 대상으로 제공됩니다. 
내년1월에는 더 많은 모델 사이즈와 함께 일반에 공개될 예정입니다.

키워드: 
1. 제미나이 
2. 구글 
3. 인공지능


### 3) 순차적인 프롬프트(Chain of Thought)를 적용합니다.
복잡한 문제를 해결할 때, 단계별로 문제를 분해하여 모델이 각 단계를 순차적으로 해결하도록 유도합니다.

In [19]:
# zero-shot 예시
response = llm.invoke(
    """
    Question: 학교에서 500명의 학생이 있습니다. 이 중 30%는 5학년이고, 20%는 6학년 학생입니다. 
              5학년 학생들 중 60%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다. 
              6학년 학생들 중 70%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다.
              과학 동아리에는 몇 명의 학생이 있나요?
    Answer:
    """
)
print(response.content)

## Step 1: Calculate the number of 5th and 6th grade students
First, let's find out how many 5th and 6th grade students there are. 30% of 500 students are 5th graders, which is \(500 \times 0.30 = 150\) students. 20% of 500 students are 6th graders, which is \(500 \times 0.20 = 100\) students.

## 2: Calculate the number of 5th grade students in math and science clubs
60% of the 5th graders are in the math club, which means \(150 \times 0.60 = 90\) students are in the math club. The remaining 5th graders are in the science club, which is \(150 - 90 = 60\) students.

## 3: Calculate the number of 6th grade students in math and science clubs
70% of the 6th graders are in the math club, which means \(100 \times 0.70 = 70\) students are in the math club. The remaining 6th graders are in the science club, which is \(100 - 70 = 30\) students.

## 4: Calculate the total number of students in the science club
To find the total number of students in the science club, we add the number of 5th an

In [21]:
# few-shot 예시
response = llm.invoke(
    """
    Question: 학교에서 300명의 학생이 있습니다. 이 중 40%는 4학년입니다. 4학년 학생들 중 절반은 축구 팀에 있고, 나머지 절반은 음악 클럽에 있습니다. 
              축구 팀에 몇 명의 학생이 있나요?
    Answer: 
    1. 첫번째 단계: 학교에는 총 300명의 학생이 있으며, 이 중 40%가 4학년입니다. 따라서, 4학년 학생 수는 전체 학생 수의 40%에 해당합니다."
    2. 두번째 단계: 4학년 학생들 중 절반은 축구 팀에 있습니다. 따라서, 축구 팀에 있는 4학년 학생 수는 4학년 학생 수의 절반에 해당합니다."
    3. 세번째 단계: 첫 번째 단계에서 구한 4학년 학생 수의 절반을 두 번째 단계의 계산으로 구합니다.
    따라서, 축구 팀에 있는 4학년 학생 수는 300 * 40% * 50% = 60명입니다.

    Question: 학교에서 500명의 학생이 있습니다. 이 중 30%는 5학년이고, 20%는 6학년 학생입니다. 
              5학년 학생들 중 60%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다. 
              6학년 학생들 중 70%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다.
              과학 동아리에는 몇 명의 학생이 있나요?
    Answer:
    """
)
print(response.content)


## Step 1: Calculate the number of 5th and 6th grade students
First, we need to find out how many 5th and 6th grade students there are. The school has 500 students in total, with 30% being 5th graders and 20% being 6th graders.
- Number of 5th graders = 500 * 30% = 500 * 0.3 = 150
- Number of 6th graders = 500 * 20% = 500 * 0.2 = 100

## 2: Calculate the number of 5th graders in the science club
60% of the 5th graders are in the math club, which means the rest, 40%, are in the science club.
- Number of 5th graders in the science club = 150 * 40% = 150 * 0.4 = 60

## 3: Calculate the number of 6th graders in the science club
70% of the 6th graders are in the math club, which means the rest, 30%, are in the science club.
- Number of 6th graders in the science club = 100 * 30% = 100 * 0.3 = 30

## 4: Calculate the total number of students in the science club
To find the total number of students in the science club, we add the number of 5th graders and 6th graders in the science club.
- To

In [20]:
# think step by step
response = llm.invoke(
    """
    Question: 학교에서 500명의 학생이 있습니다. 이 중 30%는 5학년이고, 20%는 6학년 학생입니다. 
              5학년 학생들 중 60%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다. 
              6학년 학생들 중 70%는 수학 동아리에 있고, 나머지는 과학 동아리에 있습니다.
              과학 동아리에는 몇 명의 학생이 있나요?
              Let's think step by step.
    Answer:
    """
)
print(response.content)

## Step 1: Calculate the total number of 5th and 6th-grade students.
First, we need to find out how many 5th and 6th-grade students there are in total. Given that there are 500 students in the school, 30% of them are 5th graders, and 20% are 6th graders.

## 2: Calculate the number of 5th-grade students.
30% of 500 students are 5th graders. So, the number of 5th-grade students = 30/100 * 500 = 150.

## 3: Calculate the number of 6th-grade students.
20% of 500 students are 6th graders. So, the number of 6th-grade students = 20/100 * 500 = 100.

## 4: Determine the number of 5th-grade students in the math club and science club.
60% of the 5th-grade students are in the math club, which means 60/100 * 150 = 90 students are in the math club. The remaining 5th-grade students are in the science club, so 150 - 90 = 60 students are in the science club.

## 5: Determine the number of 6th-grade students in the math club and science club.
70% of the 6th-grade students are in the math club, which m